In [1]:
from pathlib import Path

# This file's folder (notebooks/)
NB_DIR = Path.cwd()

# Repo root = parent of notebooks/
REPO_DIR = NB_DIR.parent

DATA_DIR = REPO_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

print("NB_DIR:", NB_DIR)
print("REPO_DIR:", REPO_DIR)
print("RAW_DIR:", RAW_DIR)
print("RAW_DIR exists?", RAW_DIR.exists())
print("Files:", list(RAW_DIR.glob("*"))[:10])

NB_DIR: /Users/gularhajihasanli/Downloads/ey-ai-data-water-quality-2026-dev/notebooks
REPO_DIR: /Users/gularhajihasanli/Downloads/ey-ai-data-water-quality-2026-dev
RAW_DIR: /Users/gularhajihasanli/Downloads/ey-ai-data-water-quality-2026-dev/data/raw
RAW_DIR exists? True
Files: [PosixPath('/Users/gularhajihasanli/Downloads/ey-ai-data-water-quality-2026-dev/data/raw/submission_template.csv'), PosixPath('/Users/gularhajihasanli/Downloads/ey-ai-data-water-quality-2026-dev/data/raw/.gitkeep'), PosixPath('/Users/gularhajihasanli/Downloads/ey-ai-data-water-quality-2026-dev/data/raw/water_quality_training_dataset.csv')]


In [3]:
import pandas as pd

train = pd.read_csv(RAW_DIR / "water_quality_training_dataset.csv")
submission = pd.read_csv(RAW_DIR / "submission_template.csv")

print("Train loaded")
print("Submission loaded")

Train loaded
Submission loaded


In [4]:
print("Train shape:", train.shape)
print("Submission shape:", submission.shape)

print("\nTRAIN COLUMNS:")
print(train.columns.tolist())

print("\nSUBMISSION COLUMNS:")
print(submission.columns.tolist())

Train shape: (9319, 6)
Submission shape: (200, 6)

TRAIN COLUMNS:
['Latitude', 'Longitude', 'Sample Date', 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

SUBMISSION COLUMNS:
['Latitude', 'Longitude', 'Sample Date', 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']


In [7]:
import pandas as pd

TARGETS = ["Total Alkalinity", "Electrical Conductance", "Dissolved Reactive Phosphorus"]

def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # robust date parsing for DD-MM-YYYY (and any mixed variants)
    df["Sample Date"] = pd.to_datetime(
        df["Sample Date"],
        dayfirst=True,
        errors="coerce"
    )

    # if any dates failed, fail loudly now (better than silent junk)
    bad = df["Sample Date"].isna().sum()
    if bad > 0:
        raise ValueError(f"{bad} rows have unparseable Sample Date values. Check raw CSV formatting.")

    df["year"] = df["Sample Date"].dt.year
    df["month"] = df["Sample Date"].dt.month
    df["dayofyear"] = df["Sample Date"].dt.dayofyear
    return df

train_fe = add_time_features(train)
sub_fe = add_time_features(submission)

FEATURES = ["Latitude", "Longitude", "year", "month", "dayofyear"]

X = train_fe[FEATURES]
Y = train_fe[TARGETS]
X_sub = sub_fe[FEATURES]

print("X:", X.shape, "Y:", Y.shape, "X_sub:", X_sub.shape)
print("Date range train:", train_fe["Sample Date"].min(), "->", train_fe["Sample Date"].max())

X: (9319, 5) Y: (9319, 3) X_sub: (200, 5)
Date range train: 2011-01-02 00:00:00 -> 2015-12-31 00:00:00


In [8]:
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

# Group by approximate location to avoid spatial leakage
groups = (
    train_fe["Latitude"].round(4).astype(str)
    + "_"
    + train_fe["Longitude"].round(4).astype(str)
)

gkf = GroupKFold(n_splits=5)

base = HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_depth=6,
    max_iter=600,
    random_state=42
)

model = MultiOutputRegressor(base)

maes = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, Y, groups=groups), 1):
    model.fit(X.iloc[tr_idx], Y.iloc[tr_idx])
    preds = model.predict(X.iloc[va_idx])
    fold_mae = mean_absolute_error(Y.iloc[va_idx], preds, multioutput="raw_values")
    maes.append(fold_mae)
    print(f"Fold {fold} MAE:", dict(zip(TARGETS, fold_mae)))

print("\nAvg MAE:", dict(zip(TARGETS, np.mean(maes, axis=0))))

Fold 1 MAE: {'Total Alkalinity': np.float64(54.99625360737225), 'Electrical Conductance': np.float64(251.94355690380948), 'Dissolved Reactive Phosphorus': np.float64(30.883501608148443)}
Fold 2 MAE: {'Total Alkalinity': np.float64(43.697422361731604), 'Electrical Conductance': np.float64(198.66502239711969), 'Dissolved Reactive Phosphorus': np.float64(31.306742552150975)}
Fold 3 MAE: {'Total Alkalinity': np.float64(55.302701836796615), 'Electrical Conductance': np.float64(218.01200124787826), 'Dissolved Reactive Phosphorus': np.float64(30.310474511603445)}
Fold 4 MAE: {'Total Alkalinity': np.float64(32.10043924516094), 'Electrical Conductance': np.float64(176.12149570633346), 'Dissolved Reactive Phosphorus': np.float64(33.97957557619388)}
Fold 5 MAE: {'Total Alkalinity': np.float64(56.79273812522535), 'Electrical Conductance': np.float64(177.71700522123723), 'Dissolved Reactive Phosphorus': np.float64(33.80068252297733)}

Avg MAE: {'Total Alkalinity': np.float64(48.57791103525735), 'El

In [9]:
# Train on full data
model.fit(X, Y)

# Predict submission
sub_pred = model.predict(X_sub)

out = submission.copy()
out[TARGETS] = sub_pred

# Keep column order EXACT
out = out[["Latitude", "Longitude", "Sample Date"] + TARGETS]

# Write file
out_path = REPO_DIR / "submissions" / "submission_baseline.csv"
out_path.parent.mkdir(exist_ok=True)
out.to_csv(out_path, index=False)

print("Saved submission to:", out_path)
out.head()

Saved submission to: /Users/gularhajihasanli/Downloads/ey-ai-data-water-quality-2026-dev/submissions/submission_baseline.csv


,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-32.043333,27.822778,01-09-2014,10.588967,240.965841,20.551209
1,-33.329167,26.077500,16-09-2015,132.594326,760.255605,90.719333
2,-32.991639,27.640028,07-05-2015,11.675548,210.653364,31.956053
3,-34.096389,24.439167,07-02-2012,-10.143591,402.643569,8.152287
4,-32.000556,28.581667,01-10-2014,31.437749,223.665307,21.933551


In [1]:
DATA_DIR = "../data"
RAW_DIR = f"{DATA_DIR}/raw"
PROCESSED_DIR = f"{DATA_DIR}/processed"

print("RAW_DIR set to:", RAW_DIR)

RAW_DIR set to: ../data/raw


In [2]:
import os

print("RAW_DIR =", RAW_DIR)
print("RAW_DIR exists?", os.path.exists(RAW_DIR))
print("RAW_DIR is dir?", os.path.isdir(RAW_DIR))

try:
    files = os.listdir(RAW_DIR)
    print("Files:", files)
except Exception as e:
    print("ERROR:", repr(e))

RAW_DIR = ../data/raw
RAW_DIR exists? False
RAW_DIR is dir? False
ERROR: FileNotFoundError(2, 'No such file or directory')


In [ ]:
import os

print("RAW_DIR =", RAW_DIR)
print("RAW_DIR exists?", os.path.exists(RAW_DIR))
print("RAW_DIR is dir?", os.path.isdir(RAW_DIR))

try:
    files = os.listdir(RAW_DIR)
    print("Files:", files)
except Exception as e:
    print("ERROR:", repr(e))

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.ensemble import RandomForestRegressor

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42

In [ ]:
DATA_DIR = "../data"
RAW_DIR = f"{DATA_DIR}/raw"
PROCESSED_DIR = f"{DATA_DIR}/processed"

print("Directories configured.")